In [1]:
# ! pip install selenium==4.9.0
# ! pip install msedge-selenium-tools
# ! pip install webdriver-manager

In [2]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from webdriver_manager.microsoft import EdgeChromiumDriverManager

import re
import time
from datetime import datetime
import pandas as pd

import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud

import plotly.express as px
import matplotlib.pyplot as plt

FIFA 2K24

In [31]:
id_game = 2195250

# most recent
url_template = 'https://steamcommunity.com/app/{}/reviews/?p=1&browsefilter=mostrecent&filterLanguage=english'
url = url_template.format(id_game)
url

'https://steamcommunity.com/app/2195250/reviews/?p=1&browsefilter=mostrecent&filterLanguage=english'

In [32]:
from selenium.webdriver.edge.options import Options
options = Options()
options.use_chromium = True

# Set up Edge driver
driver = webdriver.Edge(EdgeChromiumDriverManager().install())
driver.maximize_window()
driver.get(url)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10372\2744623965.py:6: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Edge(EdgeChromiumDriverManager().install())


In [33]:
# driver.quit()

In [34]:
def get_current_scroll_position(driver):
    return driver.execute_script("return window.pageYOffset;")

def scroll_to_bottom(driver):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

def get_steam_id(card):
    profile_url = card.find_element(By.XPATH, './/div[@class="apphub_friend_block"]/div/a[2]').get_attribute('href')
    steam_id = profile_url.split('/')[-2]
    return steam_id

def scrape_review_data(card):
    date_posted_element = card.find_element(By.XPATH, './/div[@class="apphub_CardTextContent"]/div[@class="date_posted"]')
    date_posted = date_posted_element.text.strip()

    # early_access_review_element = card.find_element(By.XPATH, './/div[@class="apphub_CardTextContent"]/div[@class="early_access_review"]')
    # early_access_review = early_access_review_element.text.strip()

    try:
        received_compensation_element = card.find_element(By.CLASS_NAME, "received_compensation").text
    except NoSuchElementException:
        received_compensation_element = ""

    card_text_content_element = card.find_element(By.CLASS_NAME, "apphub_CardTextContent")
    review_content = card_text_content_element.text.strip()

    excluded_elements = [date_posted, received_compensation_element]

    for excluded_element in excluded_elements:
        review_content = review_content.replace(excluded_element, "")
    review_content = review_content.replace("\n", "")

    review_length = len(review_content.replace(" ", ""))

    thumb_text = card.find_element(By.XPATH, './/div[@class="reviewInfo"]/div[2]').text
    play_hours = card.find_element(By.XPATH, './/div[@class="reviewInfo"]/div[3]').text

    return review_content, thumb_text, review_length, play_hours, date_posted



In [35]:
# reviews = []
# steam_ids_set = set()
# max_scroll_attempts = 5

# try:
#     last_position = get_current_scroll_position(driver)
#     running = True
#     while running:
#         cards = driver.find_elements(By.CLASS_NAME, "apphub_Card")

#         for card in cards[-5:]:  # This currently limits to the last 20 cards
#             steam_id = get_steam_id(card)
#             if steam_id in steam_ids_set:
#                 continue
#             else:
#                 review = scrape_review_data(card)
#                 reviews.append(review)

#         scroll_attempt = 0
#         while scroll_attempt < max_scroll_attempts:
#             scroll_to_bottom(driver)
#             curr_position = get_current_scroll_position(driver)

#             if curr_position == last_position:
#                 scroll_attempt += 1
#                 time.sleep(3)

#                 if curr_position >= 3:
#                     running = False
#                     break
#             else:
#                 last_position = curr_position
#                 break

# except Exception as e:
#     print(e)
# finally:
#     driver.quit()


In [36]:
# Scrapping
reviews = []
steam_ids_set = set()

try:
    last_position = get_current_scroll_position(driver)
    while True:  # Loop berjalan terus-menerus sampai dihentikan secara manual
        cards = driver.find_elements(By.CLASS_NAME, "apphub_Card")

        for card in cards[-5:]:  # Mengambil kartu terbaru untuk mencegah duplikasi
            steam_id = get_steam_id(card)
            if steam_id in steam_ids_set:
                continue
            else:
                review = scrape_review_data(card)
                reviews.append(review)
                steam_ids_set.add(steam_id)  # Tambahkan Steam ID ke set

        scroll_to_bottom(driver)
        curr_position = get_current_scroll_position(driver)

        if curr_position == last_position:
            time.sleep(3)  # Tunggu sebentar untuk memastikan halaman selesai dimuat
        else:
            last_position = curr_position

except KeyboardInterrupt:
    print("Scraping dihentikan oleh pengguna.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")
finally:
    driver.quit()

Scraping dihentikan oleh pengguna.


In [37]:
df = pd.DataFrame(reviews, columns=['ReviewText', 'Review', 'ReviewLength', 'PlayHours', 'DatePosted'])
df.head()

,ReviewText,Review,ReviewLength,PlayHours,DatePosted
0,its good,Recommended,7,68.9 hrs on record,Posted: 21 November
1,fubal,Recommended,5,393.8 hrs on record,Posted: 21 November
2,gg,Recommended,2,213.9 hrs on record,Posted: 21 November
3,great game,Recommended,9,310.4 hrs on record,Posted: 21 November
4,Buy if you have friends to play pro clubs with...,Not Recommended,57,248.4 hrs on record,Posted: 21 November


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2805 entries, 0 to 2804
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ReviewText    2805 non-null   object
 1   Review        2805 non-null   object
 2   ReviewLength  2805 non-null   int64 
 3   PlayHours     2805 non-null   object
 4   DatePosted    2805 non-null   object
dtypes: int64(1), object(4)
memory usage: 109.7+ KB


In [40]:
df['Review'].value_counts()

Review
Recommended        1623
Not Recommended    1179
                      3
Name: count, dtype: int64

In [ ]:
df.to_csv('FC_24.csv', index=False)
